# Nemotron SFT → GRPO Training
**Strategy**: Two-phase training on competition `train.csv`
1. **Phase 1 — SFT**: Teach the model the `<think>...</think>\boxed{answer}` format
2. **Phase 2 — GRPO**: RL fine-tuning with reward on exact/numerical boxed answer match

**Constraints**: LoRA rank ≤ 32, RTX Pro 6000 (96GB VRAM), no internet

## 0. Install Dependencies

In [ ]:
import site
import subprocess
import sys

# Add CUTLASS DSL path (Kaggle-specific)
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

# Install required packages
packages = [
    "trl>=0.12.0",
    "peft>=0.14.0",
    "accelerate>=1.0.0",
    "transformers>=4.47.0",
    "datasets>=3.0.0",
    "bitsandbytes>=0.45.0",
]
for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        check=True
    )
print("Dependencies installed.")

## 1. Imports & Config

In [ ]:
import math
import os
import re
import site

import kagglehub
import polars as pl
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH   = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
MODEL_PATH  = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
SFT_OUT     = "/kaggle/working/sft_adapter"
GRPO_OUT    = "/kaggle/working/grpo_adapter"   # final submission adapter
OUTPUT_DIR  = "/kaggle/working"

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_RANK       = 32   # max allowed by competition
LORA_ALPHA      = 64   # 2x rank is a good default
LORA_DROPOUT    = 0.05
LORA_TARGET     = r".*\.(in_proj|out_proj|up_proj|down_proj)$"  # matches submission demo

# ── SFT ──────────────────────────────────────────────────────────────────────
SFT_EPOCHS          = 2
SFT_BATCH_SIZE      = 1
SFT_GRAD_ACCUM      = 8    # effective batch = 8
SFT_LR              = 2e-4
SFT_MAX_SEQ_LEN     = 2048

# ── GRPO ─────────────────────────────────────────────────────────────────────
GRPO_STEPS          = 200
GRPO_BATCH_SIZE     = 1
GRPO_GRAD_ACCUM     = 4
GRPO_LR             = 5e-5
GRPO_NUM_GENERATIONS= 4    # G in GRPO — completions per prompt
GRPO_MAX_NEW_TOKENS = 1024
GRPO_MAX_SEQ_LEN    = 2048

print(f"Model path  : {MODEL_PATH}")
print(f"Device      : {torch.cuda.get_device_name(0)}")
print(f"VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load & Inspect Training Data

In [ ]:
df = pl.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head(3)

In [ ]:
# Inspect a sample
row = df.row(0, named=True)
print("=== PROMPT ===")
print(row["prompt"])
print("\n=== ANSWER ===")
print(row["answer"])

## 3. Load Model & Tokenizer

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for SFT loss masking

print("Loading model (bfloat16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False      # needed for gradient checkpointing
print("Model loaded.")

## 4. Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Dataset Preparation

We format each example as:
```
<user> {prompt} → put answer in \boxed{} </user>
<assistant> <think>...</think> \boxed{answer} </assistant>
```

For SFT we need complete assistant turns. Since `train.csv` only has `prompt` + `answer` (no reasoning traces), we construct a **minimal synthetic trace** that teaches the format. GRPO will then learn to reason properly via reward.

In [ ]:
BOXED_INSTRUCTION = (
    "\nPlease reason step by step. "
    "Put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

def make_sft_example(row: dict) -> dict:
    """Format a train row into a chat-template conversation for SFT.
    Assistant turn: minimal <think> block + \\boxed{answer}.
    This teaches the model the output FORMAT before GRPO teaches it to reason.
    """
    user_content = row["prompt"] + BOXED_INSTRUCTION

    # Minimal chain-of-thought stub — just enough to teach format
    # The answer goes in \boxed{} as required by the metric
    assistant_content = (
        f"<think>\nLet me solve this step by step.\n</think>\n"
        f"\\boxed{{{row['answer']}}}"
    )

    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]

    # Apply chat template to get full prompt+completion string
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        # Fallback manual format if chat template fails
        text = (
            f"User: {user_content}\n\n"
            f"Assistant: {assistant_content}"
        )
    return {"text": text, "answer": str(row["answer"])}


def make_grpo_example(row: dict) -> dict:
    """Format a train row into a prompt-only dict for GRPO.
    GRPO generates completions itself; we only supply the prompt.
    """
    user_content = row["prompt"] + BOXED_INSTRUCTION
    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        prompt = f"User: {user_content}\n\nAssistant:"
    return {"prompt": prompt, "answer": str(row["answer"])}


# Convert Polars → list of dicts → HuggingFace Dataset
rows = df.to_dicts()

sft_data  = Dataset.from_list([make_sft_example(r) for r in rows])
grpo_data = Dataset.from_list([make_grpo_example(r) for r in rows])

print(f"SFT  dataset : {len(sft_data)} examples")
print(f"GRPO dataset : {len(grpo_data)} examples")
print("\nSFT sample (first 400 chars):")
print(sft_data[0]["text"][:400])

## 6. Reward Functions (for GRPO)

Three rewards, combined additively:
- **Format reward** (+0.2): did the model produce `\boxed{...}`?
- **Correctness reward** (+1.0): does the extracted answer match ground truth?
- **Thinking reward** (+0.1): did the model produce a `<think>` block?

In [ ]:
def extract_boxed(text: str) -> str:
    """Extract the last \\boxed{} answer from model output.
    Mirrors the competition metric's extraction logic exactly.
    """
    boxed_starts = list(re.finditer(r'\\boxed\{', text))
    if not boxed_starts:
        return "NOT_FOUND"
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end = boxed_starts[i + 1].start() if i + 1 < len(boxed_starts) else len(text)
        segment = text[start:end]
        last_brace = segment.rfind("}")
        matches.append(segment[:last_brace] if last_brace != -1 else segment)
    non_empty = [m.strip() for m in matches if m.strip()]
    return non_empty[-1] if non_empty else matches[-1].strip()


def verify_answer(ground_truth: str, predicted: str) -> bool:
    """Mirrors competition metric verify() exactly."""
    ground_truth = ground_truth.strip()
    predicted    = predicted.strip()
    # Binary strings → strict
    if re.fullmatch(r'[01]+', ground_truth):
        return predicted.lower() == ground_truth.lower()
    try:
        return math.isclose(
            float(ground_truth), float(predicted),
            rel_tol=1e-2, abs_tol=1e-5
        )
    except Exception:
        return predicted.lower() == ground_truth.lower()


# ── Reward functions (each takes completions + prompts kwargs from TRL) ──────

def reward_correctness(completions, prompts, **kwargs) -> list[float]:
    """Main reward: +1.0 if boxed answer matches ground truth, else 0.0."""
    answers = kwargs["answer"]   # ground truth list, injected by GRPOTrainer
    rewards = []
    for completion, gt in zip(completions, answers):
        text      = completion[0]["content"] if isinstance(completion, list) else completion
        predicted = extract_boxed(text)
        rewards.append(1.0 if verify_answer(gt, predicted) else 0.0)
    return rewards


def reward_format(completions, **kwargs) -> list[float]:
    """Format reward: +0.2 if \\boxed{} present, else 0.0."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        has_boxed   = bool(re.search(r'\\boxed\{', text))
        rewards.append(0.2 if has_boxed else 0.0)
    return rewards


def reward_thinking(completions, **kwargs) -> list[float]:
    """Thinking reward: +0.1 if <think> block present."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        has_think = bool(re.search(r'<think>', text, re.IGNORECASE))
        rewards.append(0.1 if has_think else 0.0)
    return rewards


print("Reward functions defined.")

# Quick sanity check
test_output = "<think>\n2+2=4\n</think>\n\\boxed{4}"
print(f"extract_boxed test : {extract_boxed(test_output)!r}")
print(f"verify_answer test : {verify_answer('4', extract_boxed(test_output))}")

## 7. Phase 1 — SFT Training

In [ ]:
sft_args = SFTConfig(
    output_dir=SFT_OUT,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=SFT_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    max_seq_length=SFT_MAX_SEQ_LEN,
    dataset_text_field="text",   # field in sft_data to use as input
    # Packing: OFF — questions vary in length, packing can mix answer tokens
    packing=False,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_args,
    train_dataset=sft_data,
)

print("Starting SFT...")
sft_trainer.train()

# Save SFT adapter
model.save_pretrained(SFT_OUT)
tokenizer.save_pretrained(SFT_OUT)
print(f"SFT adapter saved to {SFT_OUT}")

## 8. Phase 2 — GRPO Training

GRPO (Group Relative Policy Optimization) generates `G` completions per prompt, computes rewards, and updates the policy to increase probability of high-reward completions relative to the group mean. No value network needed.

In [ ]:
# Reload model fresh with SFT adapter for GRPO phase
# (avoids any gradient state bleed from SFT)
print("Reloading model with SFT adapter for GRPO...")
del sft_trainer
torch.cuda.empty_cache()

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base_model.config.use_cache = False

# Load the SFT adapter weights as starting point for GRPO
grpo_model = PeftModel.from_pretrained(
    base_model,
    SFT_OUT,
    is_trainable=True,
)
grpo_model.print_trainable_parameters()
print("GRPO model ready.")

In [ ]:
grpo_config = GRPOConfig(
    output_dir=GRPO_OUT,
    max_steps=GRPO_STEPS,
    per_device_train_batch_size=GRPO_BATCH_SIZE,
    gradient_accumulation_steps=GRPO_GRAD_ACCUM,
    learning_rate=GRPO_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
    # GRPO-specific
    num_generations=GRPO_NUM_GENERATIONS,   # completions per prompt
    max_new_tokens=GRPO_MAX_NEW_TOKENS,
    max_prompt_length=GRPO_MAX_SEQ_LEN - GRPO_MAX_NEW_TOKENS,
    # KL penalty — keeps policy close to SFT reference
    beta=0.01,
    # Temperature for generation during GRPO rollout
    temperature=0.7,
    top_p=0.9,
)

grpo_trainer = GRPOTrainer(
    model=grpo_model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=grpo_data,
    reward_funcs=[
        reward_correctness,   # +1.0 — main signal
        reward_format,        # +0.2 — format compliance
        reward_thinking,      # +0.1 — encourages CoT
    ],
)

print("Starting GRPO...")
grpo_trainer.train()

# Save final adapter
grpo_model.save_pretrained(GRPO_OUT)
tokenizer.save_pretrained(GRPO_OUT)
print(f"GRPO adapter saved to {GRPO_OUT}")

## 9. Validation — Local Sanity Check

Before zipping, run a few examples through the adapter to confirm `\boxed{}` is appearing.

In [ ]:
grpo_model.eval()

def quick_inference(prompt_text: str, max_new_tokens: int = 512) -> str:
    user_content = prompt_text + BOXED_INSTRUCTION
    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        prompt = f"User: {user_content}\n\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt").to(grpo_model.device)
    with torch.no_grad():
        outputs = grpo_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,     # greedy — mirrors metric evaluation
            do_sample=False,
        )
    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Run on first 3 train examples
for i in range(min(3, len(rows))):
    row     = rows[i]
    output  = quick_inference(row["prompt"])
    pred    = extract_boxed(output)
    correct = verify_answer(str(row["answer"]), pred)
    print(f"[{i}] GT: {row['answer']!r:20s}  PRED: {pred!r:20s}  ✓" if correct else
          f"[{i}] GT: {row['answer']!r:20s}  PRED: {pred!r:20s}  ✗")
    print(f"     Output (first 200 chars): {output[:200]}")
    print()

## 10. Package Submission

In [ ]:
import os
import shutil
import subprocess

# Copy GRPO adapter files into /kaggle/working root
# The metric server searches for adapter_config.json recursively,
# but keeping it at root is cleanest.
for fname in os.listdir(GRPO_OUT):
    src = os.path.join(GRPO_OUT, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    shutil.copy2(src, dst)
    print(f"Copied: {fname}")

# Verify adapter_config.json exists — required by metric
assert os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")), \
    "ERROR: adapter_config.json missing! Submission will be rejected."
print("\nadapter_config.json ✓")

# Zip everything
submission_path = os.path.join(OUTPUT_DIR, "submission.zip")
subprocess.run(
    f"cd {OUTPUT_DIR} && zip -m submission.zip adapter_config.json adapter_model* tokenizer*",
    shell=True, check=True
)
size_mb = os.path.getsize(submission_path) / 1e6
print(f"\nsubmission.zip created: {size_mb:.1f} MB")
print("Done. Submit /kaggle/working/submission.zip to Kaggle.")

## Appendix: Key Design Decisions

| Decision | Choice | Reason |
|---|---|---|
| LoRA rank | 32 | Competition maximum |
| LoRA alpha | 64 | 2× rank — standard scaling |
| Target modules | in/out/up/down_proj | Matches submission demo |
| SFT format | `<think>stub</think>\\boxed{answer}` | Teaches output format before RL |
| GRPO β (KL) | 0.01 | Light regularization — SFT already close |
| GRPO generations | 4 | Balance variance vs VRAM |
| Rewards | Correctness + Format + Thinking | Multi-signal, weighted by importance |
| Rollout temperature | 0.7 | Diversity during GRPO rollouts |
| Eval temperature | 0.0 | Mirrors metric server (greedy) |